# The four tool-design anti-patterns, named

```yaml
title: "The four tool-design anti-patterns, named"
keywords: tool soup, over-tooling, tool-design anti-patterns, vague description, misleading description, opaque errors, god-tool, loose schema, tool selection, bind_tools, FakeModel, Sonnet 4.6, Haiku 4.5, anthropic, claude
estimated duration: 12
```

<a id="top"></a>

> **Lesson:** L08. Teacher demo — Demo 5 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L08/demos_or_activities.md), the **anti-pattern capstone**. It closes the arc of Demos 1–4 ([L0803](L0803_lecture.ipynb) tool-or-no-tool, [L0805](L0805_lecture.ipynb) the description, [L0807](L0807_lecture.ipynb) schemas + errors, [L0809](L0809_lecture.ipynb) side effects) by naming the **four ways a tool set silently makes the agent worse**.
> **Anchor model: Claude Sonnet 4.6**, with a Haiku 4.5 contrast (a smaller model degrades sooner).
>
> **Runs two ways.** By default this notebook drives the offline scripted `FakeModel` (no key, deterministic, restart-and-run-all passes). Set `ANTHROPIC_API_KEY` and the one optional live cell drives real Sonnet 4.6 → Haiku 4.5. The client seam is LangChain's `ChatAnthropic` + `bind_tools` (from L03/L07); the key loads through the config seam (`require_anthropic_key`), never hard-coded.

## Contents

- [1. Setup](#1-setup)
- [2. Tool soup (the one new beat)](#2-tool-soup-the-one-new-beat)
- [3. The other three, named](#3-the-other-three-named)
- [4. Tie back to the payoff](#4-tie-back-to-the-payoff)
- [5. Takeaways](#5-takeaways)

## 1. Setup

The four earlier demos each built one *good* design move. This capstone names the four ways a tool
set silently makes the agent **worse** — the mirror image of Demos 1–4. Only one of them is genuinely
new here: **tool soup** (too many overlapping tools), which we run live. The other three are the
bad-design variants you already watched break, brought back to be *named*.

The setup cell below loads the shared seams: LangChain message types, the offline `FakeModel` (from
`common.fake_model`, so a keyless restart-and-run-all passes), and the config seam for the optional
live path. `LIVE` is `True` only when `ANTHROPIC_API_KEY` is set.

In [1]:
import json
from typing import Annotated

from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.utils.function_calling import convert_to_openai_tool

from fluffy_potato_curriculum.common.config import get_settings, require_anthropic_key
from fluffy_potato_curriculum.common.fake_model import FakeModel, tool_call, tool_reply

SONNET = "claude-sonnet-4-6"  # course anchor
HAIKU = "claude-haiku-4-5"  # smaller model -- selection degrades sooner

# Offline is the default. Set ANTHROPIC_API_KEY to light up the optional live cell.
LIVE = bool(get_settings().anthropic_api_key)

# One request that several overlapping tools all plausibly match.
REQUEST = "What's Alex's email?"


def picked_tools(resp: AIMessage) -> list[str]:
    """The names of the tools the model proposed this turn (possibly empty)."""
    return [call["name"] for call in resp.tool_calls]


def show_turn(resp: AIMessage) -> None:
    """Print a model turn: any text, plus each tool call (name + args)."""
    print("tool_calls:", picked_tools(resp))
    if resp.content:
        print("  text:", repr(resp.content))
    for call in resp.tool_calls:
        args = ", ".join(f"{k}={v!r}" for k, v in call["args"].items())
        print(f"  call  {call['name']}({args})")


print("setup ready -- LIVE =", LIVE)

setup ready -- LIVE = False


[↑ Back to top](#top)

## 2. Tool soup (the one new beat)

**Over-tooling** has two faces. The first you already met in [Demo 1](L0803_lecture.ipynb): a tool
for what the model does well alone is pure overhead *and* a wrong-tool option it can pick by mistake.
The second is new — **tool soup**: several tools whose descriptions overlap so much that more than
one plausibly matches the same request. Now the model has to *choose*, and it has nothing clean to
choose on.

Below is a "soup" registry: six near-identical user-lookup tools plus two unrelated distractors.
Their descriptions deliberately overlap, and all six return the *same* record — so which one fires is
arbitrary. That is the soup.

In [2]:
ALEX = {"name": "Alex Kim", "email": "alex@example.com"}


# Six tools that all plausibly answer "what's Alex's email?" -- the descriptions overlap
# so the model has no clean way to choose, and all six return the SAME record.
def lookup_user(query: Annotated[str, "a name, email, or username"]) -> dict[str, str]:
    """Look up a user by name, email, or username."""
    return ALEX


def find_user(name: Annotated[str, "the user's name or handle"]) -> dict[str, str]:
    """Find a user account by name or identifier."""
    return ALEX


def get_customer(identifier: Annotated[str, "any customer identifier"]) -> dict[str, str]:
    """Get a customer record by any identifier."""
    return ALEX


def search_accounts(term: Annotated[str, "a search term"]) -> dict[str, str]:
    """Search accounts matching a term and return the best match."""
    return ALEX


def user_info(who: Annotated[str, "who to describe"]) -> dict[str, str]:
    """Return information about a user."""
    return ALEX


def whois(handle: Annotated[str, "a handle or address to resolve"]) -> dict[str, str]:
    """Look up who a handle or address belongs to."""
    return ALEX


# Two unrelated distractors -- they should never match a "user email" request.
def convert_currency(
    amount: Annotated[float, "the amount to convert"],
    to_currency: Annotated[str, "the ISO currency code to convert to"],
) -> str:
    """Convert a money amount from USD into another currency."""
    return f"{amount} USD -> {to_currency}"


def get_weather(city: Annotated[str, "a city name"]) -> str:
    """Get the current weather for a city."""
    return f"sunny in {city}"


SOUP_TOOLS = [
    lookup_user,
    find_user,
    get_customer,
    search_accounts,
    user_info,
    whois,
    convert_currency,
    get_weather,
]

print(f"the soup: {len(SOUP_TOOLS)} tools, all always-on for every request\n")
for fn in SOUP_TOOLS:
    print(f"  {fn.__name__:16s} {fn.__doc__}")

the soup: 8 tools, all always-on for every request

  lookup_user      Look up a user by name, email, or username.
  find_user        Find a user account by name or identifier.
  get_customer     Get a customer record by any identifier.
  search_accounts  Search accounts matching a term and return the best match.
  user_info        Return information about a user.
  whois            Look up who a handle or address belongs to.
  convert_currency Convert a money amount from USD into another currency.
  get_weather      Get the current weather for a city.


In [3]:
# Every registered tool's schema rides in the request on EVERY turn -- this is L07's
# "tokens twice over": you pay for the schema whether or not the tool is ever used.
# So more tools is a bigger always-on bill, before selection is even considered.
soup_chars = sum(len(json.dumps(convert_to_openai_tool(fn))) for fn in SOUP_TOOLS)
one_chars = len(json.dumps(convert_to_openai_tool(lookup_user)))

print(f"8-tool soup schema: ~{soup_chars} chars always-on, every request")
print(f"1-tool version:     ~{one_chars} chars")
print(f"the soup carries ~{soup_chars / one_chars:.0f}x the always-on schema budget")

8-tool soup schema: ~2195 chars always-on, every request
1-tool version:     ~266 chars
the soup carries ~8x the always-on schema budget


Now the selection itself. **Offline is the default**, so the picks below are *scripted* with `FakeModel` to
keep the run reproducible with no key — but keep the caveat in mind: on a live model these picks are
genuinely non-deterministic. That is exactly the failure. Across three runs of the *same* request the
scripted model reaches for a **different** overlapping tool each time, and once redundantly **chains
two** of them.

In [ ]:
# Scripted for reproducibility: runs 1-2 pick DIFFERENT overlapping tools; run 3
# redundantly chains two. On a live model this instability is real, not scripted.
soup_runs = FakeModel(
    [
        tool_reply(tool_call("c1", "get_customer", {"identifier": "Alex"})),
        tool_reply(tool_call("c2", "user_info", {"who": "Alex"})),
        tool_reply(
            tool_call("c3", "find_user", {"name": "Alex"}),
            tool_call("c4", "search_accounts", {"term": "Alex"}),
        ),
    ]
)
bound = soup_runs.bind_tools(SOUP_TOOLS)

print("request:", REQUEST)
for i in range(3):
    resp = await bound.ainvoke([HumanMessage(REQUEST)])
    print(f"\n=== run {i + 1} ===")
    show_turn(resp)
print("\nsame request, three different choices -> selection is unstable across the set")

**Cure:** the *fewest* tools that cover the job. The six overlapping lookups all did the same thing,
so they collapse into **one** well-named tool — nothing left to dither over, and the pick is stable
and traceable across runs. When two lookups genuinely differ, keep the descriptions **mutually
distinct** (the [Demo 2](L0805_lecture.ipynb) discipline) so only one fires per request.

In [ ]:
# Collapse the six overlapping lookups into one. With a single tool there is nothing
# to choose between, so the pick is stable every run.
one_tool = FakeModel([tool_reply(tool_call("c1", "lookup_user", {"query": "Alex"}))])
bound = one_tool.bind_tools([lookup_user])

for i in range(3):
    resp = await bound.ainvoke([HumanMessage(REQUEST)])
    print(f"run {i + 1}:", picked_tools(resp))
print("one tool, one unambiguous choice, every run")

The scripted runs decided the picks in advance so the offline notebook is reproducible. On a *live*
model the wrong or shifting pick is genuinely non-deterministic, and a **smaller / cheaper** model
degrades **sooner** than Sonnet 4.6 — the same model-class contrast as
[L07 Demo 4](../L07/L0708_lecture.ipynb) and [L0805](L0805_lecture.ipynb)'s Haiku re-run. The guarded
cell below runs the soup against real Sonnet 4.6, then Haiku 4.5, only if a key is set.

In [ ]:
# OPTIONAL live path -- runs only if ANTHROPIC_API_KEY is set. Keyless runs skip it,
# so restart-and-run-all still passes offline. Selection is non-deterministic here.
if LIVE:
    from langchain_anthropic import ChatAnthropic

    key = require_anthropic_key()
    for model_id in (SONNET, HAIKU):
        model = ChatAnthropic(model=model_id, api_key=key, max_tokens=200)
        bound = model.bind_tools(SOUP_TOOLS)
        picks: list[list[str]] = []
        for _ in range(3):
            picks.append(picked_tools(await bound.ainvoke([HumanMessage(REQUEST)])))
        print(f"{model_id:20s} picked across 3 runs: {picks}")
else:
    print("LIVE is False -- skipping the real-model run (set ANTHROPIC_API_KEY to enable).")

[↑ Back to top](#top)

## 3. The other three, named

The remaining three are not new — you watched each break in Demos 2–4. Here we only **name** them and
state the one-line cure, so you carry the full set as a checklist. The cell below brings the exact
bad-design variants back on screen; the naming is the deliverable.

In [7]:
# The exact bad-design variants from Demos 2-4, recalled so we can NAME each. Nothing
# new is computed here -- this is the recap, one cure per anti-pattern.

# From L0805 (Demo 2): the same lookup tool with a vague and an overstated description.
SPARSE_DESC = "Looks up a user."
MISLEADING_DESC = (
    "Looks up any information about any user -- email, billing, preferences, and full "
    "account history. Call this for anything user-related."
)  # the tool actually returns only {name, email} -- the description overstates it.

# From L0807 (Demo 3): an opaque validation error vs. the informative one.
OPAQUE_ERROR = {"error": "bad input"}
INFORMATIVE_ERROR = {
    "error": "validation",
    "field": "duration_minutes",
    "message": "must be an integer 15..240, got 500",
}

# From L0807 (Demo 3): the god-tool -- one free-form string vs. a typed, validated schema.
GOD_TOOL_SIG = "book_meeting(details: str)"
TYPED_SIG = "book_meeting(attendee_email: str, start_iso: str, duration_minutes: int, title: str)"

print("2. vague / misleading description")
print("   bad (sparse):    ", SPARSE_DESC)
print("   bad (overstated):", MISLEADING_DESC[:58], "...")
print("\n3. unhandled / opaque error")
print("   bad: ", OPAQUE_ERROR)
print("   good:", INFORMATIVE_ERROR)
print("\n4. the god-tool")
print("   bad: ", GOD_TOOL_SIG)
print("   good:", TYPED_SIG)

2. vague / misleading description
   bad (sparse):     Looks up a user.
   bad (overstated): Looks up any information about any user -- email, billing, ...

3. unhandled / opaque error
   bad:  {'error': 'bad input'}
   good: {'error': 'validation', 'field': 'duration_minutes', 'message': 'must be an integer 15..240, got 500'}

4. the god-tool
   bad:  book_meeting(details: str)
   good: book_meeting(attendee_email: str, start_iso: str, duration_minutes: int, title: str)


### 3.1 Vague or misleading description

The description is the model's only selection signal ([L0805](L0805_lecture.ipynb)). **Vague**
(`"Looks up a user."`) → the tool is skipped or called with garbage args. **Overstated** → confident
calls for data the tool can't return, then a confused follow-up. **Cure:** say *when to call* and
*what comes back*; write for the model's selection step, not the human code reader.

### 3.2 Unhandled or opaque errors

An error the model can't read becomes a blind retry loop ([L0807](L0807_lecture.ipynb)):
`{"error": "bad input"}` names no field and no fix. **Cure:** errors are part of the interface — name
the field and the constraint. An error message is a *prompt* for the model's next turn.

### 3.3 The god-tool

One tool doing everything behind a free-form `details: str` ([L0807](L0807_lecture.ipynb)) hides
wrongness and can't be validated — the "too-generic" extreme from
[L0802](L0802_lecture.md) §4.4. **Cure:** split by responsibility; give each a tight, typed schema so
ambiguity surfaces as a recoverable validation error instead of a silent bad write.

[↑ Back to top](#top)

## 4. Tie back to the payoff

A tool set earns its place only when it makes the model **more reliable and cheaper to reason about**
than model-alone. Each anti-pattern is one of Demos 1–4's good moves *inverted*:

| Good design (Demos 1–4) | Inverted into the anti-pattern |
| --- | --- |
| Add a tool only when it beats model-alone (Demo 1) | **Tool soup** — many overlapping tools; selection degrades and the schema bill climbs |
| A description written for the model's selection step (Demo 2) | **Vague / misleading description** — the model can't tell when to call it |
| Errors shaped as a prompt for the next turn (Demo 3) | **Unhandled / opaque errors** — the model can't recover |
| A tight, typed schema per responsibility (Demo 3) | **The god-tool** — one free-form string hides wrongness |

Two of these scale forward almost verbatim: **description collisions** and "a tool that should be
something else" are exactly [L22](../L22/L2201_intro.md)/[L23](../L23/L2301_intro.md)'s skill
anti-patterns — the same failure, one level up. You'll see the link again there; this capstone doesn't
re-teach it.

[↑ Back to top](#top)

## 5. Takeaways

- **A tool set is a win only when it makes the model more reliable and cheaper to reason about than
  model-alone.** These four are the ways it silently doesn't — the mirror image of Demos 1–4. Naming
  them as a checklist is the point of this capstone.
- **Tool soup** (the one new failure) — too many overlapping tools; selection is unstable *and* every
  schema rides in every request (L07's "tokens twice over"). *Cure:* the fewest tools that cover the
  job; merge overlaps; keep descriptions **mutually distinct** (Demo 2) — the discipline that becomes
  skill-description collisions in [L22](../L22/L2201_intro.md)/[L23](../L23/L2301_intro.md).
- **Vague / misleading description** — the model's only selection signal is unclear or overstated.
  *Cure:* say when to call and what comes back; write for the model, not the code reader.
- **Unhandled / opaque errors** — an unreadable error becomes a blind retry loop. *Cure:* name the
  field and the constraint; an error is a prompt for the next turn.
- **The god-tool** — one free-form string does everything and hides wrongness. *Cure:* split by
  responsibility; one tight, typed schema each.
- The soup beat is sharpest on a **smaller model**: Sonnet 4.6 may select cleanly where Haiku 4.5
  dithers — robustness rises with model power and falls with tool count ([L07 Demo 4](../L07/L0708_lecture.ipynb)'s contrast).

[↑ Back to top](#top)